# NCO tract lengths and positioning


## Setup


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import glob
import io
import os
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import scipy.ndimage
import scipy.stats
import seaborn as sns
import statsmodels.stats.proportion
import tqdm
from brokenaxes import brokenaxes

repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent
elif not (repo / "notebooks").exists():
    repo = Path("/nfs/users/nfs_r/rs42/rs42/git/recombination")

sys.path.append(str(repo))
from src.IDs import *

figure_dir = repo / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)

plt.rcParams["pdf.use14corefonts"] = True
pl.Config.set_tbl_rows(-1)
pl.Config.set_fmt_str_lengths(50)

CO_color = "#4C78A8"
NCO_color = "#54A24B"


In [ ]:
T2T_chromosome_sizes_in_bp = {
    "chr1": 248387328,
    "chr2": 242696752,
    "chr3": 201105948,
    "chr4": 193574945,
    "chr5": 182045439,
    "chr6": 172126628,
    "chr7": 160567428,
    "chr8": 146259331,
    "chr9": 150617247,
    "chr10": 134758134,
    "chr11": 135127769,
    "chr12": 133324548,
    "chr13": 113566686,
    "chr14": 101161492,
    "chr15": 99753195,
    "chr16": 96330374,
    "chr17": 84276897,
    "chr18": 80542538,
    "chr19": 61707364,
    "chr20": 66210255,
    "chr21": 45090682,
    "chr22": 51324926,
    "chrX": 154259566,
    "chrY": 62460029,
}


## Events


In [ ]:
rahbari_df = pl.read_csv(repo / "configs" / "Rahbari.tsv", separator="\t")
sudmant_df = (
    pl.read_csv(repo / "configs" / "Sudmant.tsv", separator="\t")
    .with_columns(
        pl.col("sample_set").cast(pl.String),
        pl.col("sample_id").cast(pl.String),
    )
)

rahbari_output = Path("/lustre/scratch122/tol/projects/sperm/results/Rahbari_20250212")
sudmant_output = Path("/lustre/scratch122/tol/projects/sperm/results/Sudmant_20241121")

read_fields = [
    "read_name",
    "read_length",
    "chrom",
    "sample_id",
    "high_quality_snp_positions",
    "high_quality_snp_positions_alleles",
    "CO_active_interval_crossover_prob",
    "NCO_active_interval_crossover_prob",
    "NCO_prob_detection_in_CO_active_interval",
    "full_read_crossover_prob",
    "grch37_reference_start",
    "grch38_reference_start",
    "grch37_reference_end",
    "grch38_reference_end",
    "T2T_reference_start",
    "AA_motif_center_pos",
    "AA_heat",
    "AA_motif_strand",
    "is_high_quality_read",
    "is_contamination",
    "high_quality_classification_class",
    "high_quality_classification_in_detectable_class",
    "snp_positions_on_read",
    "idx_transitions",
]

def read_paths(config_df, output_path):
    for sample_id, sample_set in config_df.select("sample_id", "sample_set").unique().iter_rows():
        for chrom in aut_chrom_names:
            yield output_path / "read_analysis" / sample_set / sample_id / "reads" / chrom / "all_reads_structure_annotated.parquet"

def scan_co_nco(config_df, output_path):
    return pl.concat([
        pl.scan_parquet(str(filename))
        .select(read_fields)
        .filter(pl.col("high_quality_classification_class").is_in(["CO", "GC"]))
        .filter(pl.col("is_high_quality_read"))
        .filter(~pl.col("is_contamination"))
        for filename in read_paths(config_df, output_path)
    ])

CO_NCO_df = (
    pl.concat([
        scan_co_nco(rahbari_df, rahbari_output),
        scan_co_nco(sudmant_df, sudmant_output),
    ])
    .collect()
    .filter(pl.col("high_quality_snp_positions").list.len() >= 3)
    .filter(pl.col("CO_active_interval_crossover_prob") > 0)
    .with_columns(
        genetic_length_in_cm=pl.col("full_read_crossover_prob") * 1e2,
        genetic_length_in_bp=pl.col("read_length"),
        lower_bound=(
            pl.col("snp_positions_on_read").list.get(pl.col("idx_transitions").list.get(1, null_on_oob=True)) -
            pl.col("snp_positions_on_read").list.get(pl.col("idx_transitions").list.get(0, null_on_oob=True) + 1)
        ),
        upper_bound=(
            pl.col("snp_positions_on_read").list.get(pl.col("idx_transitions").list.get(1, null_on_oob=True) + 1) -
            pl.col("snp_positions_on_read").list.get(pl.col("idx_transitions").list.get(0, null_on_oob=True))
        ),
        n_converted=pl.col("idx_transitions").list.diff(null_behavior="drop").list.first(),
    )
    .with_columns(
        read_recomb_rate_in_cm_bp=pl.col("genetic_length_in_cm") / (pl.col("genetic_length_in_bp") * 1e-6)
    )
)

NCO_read_df = CO_NCO_df.filter(pl.col("high_quality_classification_class") == "GC")
NCO_clean_df = NCO_read_df.filter(pl.col("high_quality_classification_in_detectable_class") == "NCO")
CO_NCO_df.group_by("high_quality_classification_class").len()


## Tract lengths


In [ ]:
events_df = pl.read_parquet(
    "/lustre/scratch122/tol/projects/sperm/results/recombination_events_sperm_20250325.parquet"
)

def event_tract_stats(row):
    positions = row["snp_positions"]
    alleles = row["snp_alleles"]
    background = alleles[0]
    converted_idx = [i for i, allele in enumerate(alleles) if allele != background]
    if len(converted_idx) == 0:
        return (0, None, None)
    first = converted_idx[0]
    last = converted_idx[-1]
    return (
        len(converted_idx),
        positions[last] - positions[first],
        positions[min(last + 1, len(positions) - 1)] - positions[max(first - 1, 0)],
    )

NCO_tract_df = pl.DataFrame(
    [event_tract_stats(row) for row in events_df.filter(pl.col("event_type") == "NCO").iter_rows(named=True)],
    schema=["n_converted", "lower_bound", "upper_bound"],
    orient="row",
)

NCO_tract_df.group_by("n_converted").len().sort("n_converted")


In [ ]:
tract_dir = Path("/lustre/scratch122/tol/projects/sperm/results/tract_length_inference_sperm_20250407")
sim_stats_2comp_df = pl.read_parquet(tract_dir / "tract_stats_m=0.993_L1=31.000_L2=1220.000.parquet")
sim_stats_1comp_df = pl.read_parquet(tract_dir / "tract_stats_m=1.000_L1=71.503_L2=1000.000.parquet")
bootstraps = pl.read_parquet(tract_dir / "bootstrap_opts.parquet")
bootstraps_singles = pl.read_parquet(tract_dir / "bootstrap_opts_single.parquet")

mixture_ci = np.quantile(np.vstack(bootstraps["params"].to_list()), q=[0.025, 0.975], axis=0)
single_ci = np.quantile(np.vstack(bootstraps_singles["params"].to_list()[0]), q=[0.025, 0.975], axis=0)


In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))

observed_sorted = NCO_tract_df.filter(pl.col("lower_bound") > 0)["lower_bound"].sort()
single_sorted = sim_stats_1comp_df.filter(pl.col("lower_bound") > 0)["lower_bound"].sort()
mixture_sorted = sim_stats_2comp_df.filter(pl.col("lower_bound") > 0)["lower_bound"].sort()

observed_ecdf = np.arange(1, len(observed_sorted) + 1) / len(observed_sorted)
single_ecdf = np.arange(1, len(single_sorted) + 1) / len(single_sorted)
mixture_ecdf = np.arange(1, len(mixture_sorted) + 1) / len(mixture_sorted)

ax.plot(mixture_sorted, mixture_ecdf, color="C0", linestyle="-", label="Short + Long components", linewidth=1)
ax.plot(single_sorted, single_ecdf, color="C1", linestyle="-", label="Single component", linewidth=1)
ax.plot(observed_sorted, observed_ecdf, color=NCO_color, label="NCO reads", linewidth=2)

ax.set_xscale("log")
ax.set_xlim(10**0.8, 1e4)
ax.set_xlabel("Distance between converted SNPs (bp)")
ax.set_ylabel("Cumulative proportion")
ax.legend()
plt.grid(True, which="both", ls="--", linewidth=0.5)
fig.tight_layout()
fig.savefig(figure_dir / "nco_tract_length_cdf.pdf")


In [ ]:
total_n_converted = len(NCO_tract_df["n_converted"])
mns = np.histogram(NCO_tract_df["n_converted"], bins=np.arange(1, 6) - 1e-4)[0] / total_n_converted

total_conv_2comp = len(sim_stats_2comp_df["n_converted"])
n_conv_mns_2comp = np.histogram(sim_stats_2comp_df["n_converted"], bins=np.arange(1, 6))[0] / total_conv_2comp

total_conv_1comp = len(sim_stats_1comp_df["n_converted"])
n_conv_mns_1comp = np.histogram(sim_stats_1comp_df["n_converted"], bins=np.arange(1, 6))[0] / total_conv_1comp

fig = plt.figure(figsize=(4.5, 4))
bax = brokenaxes(xlims=[(0.4, 4.6)], ylims=[(0, 0.15), (0.85, 0.95)], hspace=.4)

bax.plot(
    np.arange(len(n_conv_mns_2comp)) + 1,
    n_conv_mns_2comp,
    ".",
    color="C0",
    label="Short + Long components",
    ms=10,
)
bax.plot(
    np.arange(len(n_conv_mns_1comp)) + 1,
    n_conv_mns_1comp,
    ".",
    color="C1",
    label="Single component",
    ms=10,
)

ci_high = scipy.stats.binom.isf(0.025, total_n_converted, mns) / total_n_converted
ci_low = scipy.stats.binom.isf(1 - 0.025, total_n_converted, mns) / total_n_converted
bax.bar(
    x=np.arange(len(mns)) + 1,
    height=mns,
    yerr=np.maximum(0, np.array([[mn - lo, hi - mn] for mn, lo, hi in zip(mns, ci_low, ci_high)]).T),
    color=NCO_color,
    alpha=0.6,
    label="NCO reads",
    edgecolor="black",
    linewidth=1,
    capsize=5,
    ecolor="gray",
)

bax.set_xlabel("# of converted markers")
bax.set_ylabel("Proportion")
bax.legend()
fig.savefig(figure_dir / "nco_converted_marker_count_fit.pdf")


## Telomeres


In [ ]:
telomere_cutoff = 20e6
telomere_step = telomere_cutoff / 20
min_snps = 3
limited_sample_ids = rahbari_sample_ids + sudmant_sample_ids
acrocentric_chroms = ["chr13", "chr14", "chr15", "chr21", "chr22"]

p_arm_distances = (
    CO_NCO_df
    .filter(pl.col("sample_id").is_in(limited_sample_ids))
    .filter(pl.col("high_quality_snp_positions").list.len() >= min_snps)
    .select(
        "read_name",
        dist_from_p_telomere=pl.when(~pl.col("chrom").is_in(acrocentric_chroms)).then(
            pl.col("T2T_reference_start") + pl.col("read_length") // 2
        ),
    )
)

q_arm_distances = (
    CO_NCO_df
    .filter(pl.col("sample_id").is_in(limited_sample_ids))
    .filter(pl.col("high_quality_snp_positions").list.len() >= min_snps)
    .join(pl.DataFrame(list(T2T_chromosome_sizes_in_bp.items()), schema=["chrom", "T2T_chrom_length"], orient="row"), on="chrom")
    .select(
        "read_name",
        dist_from_q_telomere=pl.col("T2T_chrom_length") - (pl.col("T2T_reference_start") + pl.col("read_length") // 2),
    )
)

close_to_telomeres_df = (
    CO_NCO_df
    .filter(pl.col("sample_id").is_in(limited_sample_ids))
    .filter(pl.col("high_quality_snp_positions").list.len() >= min_snps)
    .join(p_arm_distances, on="read_name")
    .join(q_arm_distances, on="read_name")
    .with_columns(distance_to_telomere=pl.min_horizontal("dist_from_p_telomere", "dist_from_q_telomere"))
    .filter(pl.col("distance_to_telomere") < telomere_cutoff)
)

high_qual_CO_telomere_exp_probs = (
    close_to_telomeres_df
    .group_by(pl.col("distance_to_telomere") // telomere_step)
    .agg(pl.col("CO_active_interval_crossover_prob").mean().alias("prob"))
    .sort("distance_to_telomere")
)["prob"]
high_qual_CO_telomere_exp_freqs = high_qual_CO_telomere_exp_probs / high_qual_CO_telomere_exp_probs.sum()

high_qual_NCO_telomere_exp_probs = (
    close_to_telomeres_df
    .group_by(pl.col("distance_to_telomere") // telomere_step)
    .agg((pl.col("NCO_active_interval_crossover_prob") * pl.col("NCO_prob_detection_in_CO_active_interval")).mean().alias("prob"))
    .sort("distance_to_telomere")
)["prob"]
high_qual_NCO_telomere_exp_freqs = high_qual_NCO_telomere_exp_probs / high_qual_NCO_telomere_exp_probs.sum()

bins = np.arange(0, telomere_cutoff + telomere_step, telomere_step)

high_qual_CO_telomere_counts = np.histogram(
    close_to_telomeres_df.filter(pl.col("high_quality_classification_in_detectable_class") == "CO")["distance_to_telomere"],
    bins=bins,
)[0]
high_qual_CO_telomere_freqs = high_qual_CO_telomere_counts / high_qual_CO_telomere_counts.sum()

high_qual_NCO_telomere_counts = np.histogram(
    close_to_telomeres_df.filter(pl.col("high_quality_classification_in_detectable_class") == "NCO")["distance_to_telomere"],
    bins=bins,
)[0]
high_qual_NCO_telomere_freqs = high_qual_NCO_telomere_counts / high_qual_NCO_telomere_counts.sum()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))

ax.plot(bins[:-1], high_qual_CO_telomere_freqs, ".-", label="COs", ms=10, color=CO_color)
ax.plot(bins[:-1], high_qual_CO_telomere_exp_freqs, ".--", label="COs (expected)", ms=5, color=CO_color, alpha=0.5)
ax.plot(bins[:-1], high_qual_NCO_telomere_freqs, ".-", label="NCOs", ms=10, color=NCO_color)
ax.plot(bins[:-1], high_qual_NCO_telomere_exp_freqs, ".--", label="NCOs (expected)", ms=5, color=NCO_color, alpha=0.5)

ax.legend()
ax.set_xticks(bins[:-1], labels=[f"{int(x / 1e6)}" for x in bins[:-1]])
ax.set_xlabel("Distance to telomere (Mb)")
ax.set_ylabel("Proportion")
ax.spines[["right", "top"]].set_visible(False)
fig.tight_layout()
fig.savefig(figure_dir / "co_nco_telomere_distance_distribution.pdf")


## Dataset split

In [ ]:
def telomere_distribution(sample_ids):
    p_arm_distances = (
        CO_NCO_df
        .filter(pl.col("sample_id").is_in(sample_ids))
        .filter(pl.col("high_quality_snp_positions").list.len() >= min_snps)
        .select(
            "read_name",
            dist_from_p_telomere=pl.when(~pl.col("chrom").is_in(acrocentric_chroms)).then(
                pl.col("T2T_reference_start") + pl.col("read_length") // 2
            ),
        )
    )
    q_arm_distances = (
        CO_NCO_df
        .filter(pl.col("sample_id").is_in(sample_ids))
        .filter(pl.col("high_quality_snp_positions").list.len() >= min_snps)
        .join(pl.DataFrame(list(T2T_chromosome_sizes_in_bp.items()), schema=["chrom", "T2T_chrom_length"], orient="row"), on="chrom")
        .select(
            "read_name",
            dist_from_q_telomere=pl.col("T2T_chrom_length") - (pl.col("T2T_reference_start") + pl.col("read_length") // 2),
        )
    )
    close_df = (
        CO_NCO_df
        .filter(pl.col("sample_id").is_in(sample_ids))
        .filter(pl.col("high_quality_snp_positions").list.len() >= min_snps)
        .join(p_arm_distances, on="read_name")
        .join(q_arm_distances, on="read_name")
        .with_columns(distance_to_telomere=pl.min_horizontal("dist_from_p_telomere", "dist_from_q_telomere"))
        .filter(pl.col("distance_to_telomere") < telomere_cutoff)
    )
    co_exp = (
        close_df
        .group_by(pl.col("distance_to_telomere") // telomere_step)
        .agg(pl.col("CO_active_interval_crossover_prob").mean().alias("prob"))
        .sort("distance_to_telomere")
    )["prob"]
    nco_exp = (
        close_df
        .group_by(pl.col("distance_to_telomere") // telomere_step)
        .agg((pl.col("NCO_active_interval_crossover_prob") * pl.col("NCO_prob_detection_in_CO_active_interval")).mean().alias("prob"))
        .sort("distance_to_telomere")
    )["prob"]
    co_counts = np.histogram(
        close_df.filter(pl.col("high_quality_classification_in_detectable_class") == "CO")["distance_to_telomere"],
        bins=bins,
    )[0]
    nco_counts = np.histogram(
        close_df.filter(pl.col("high_quality_classification_in_detectable_class") == "NCO")["distance_to_telomere"],
        bins=bins,
    )[0]
    return {
        "co_freqs": co_counts / co_counts.sum(),
        "nco_freqs": nco_counts / nco_counts.sum(),
        "co_exp_freqs": co_exp / co_exp.sum(),
        "nco_exp_freqs": nco_exp / nco_exp.sum(),
        "co_counts": co_counts,
        "nco_counts": nco_counts,
    }

fig, axs = plt.subplots(1, 2, figsize=(8, 3), sharey=True)
subtelomere_summary = {}
for ax, label, sample_ids in zip(axs, ["TwinsUK", "SL"], [rahbari_sample_ids, sudmant_sample_ids]):
    d = telomere_distribution(sample_ids)
    subtelomere_summary[label] = {
        "CO nearest bin": int(d["co_counts"][0]),
        "NCO nearest bin": int(d["nco_counts"][0]),
        "CO total": int(d["co_counts"].sum()),
        "NCO total": int(d["nco_counts"].sum()),
    }
    ax.plot(bins[:-1], d["co_freqs"], ".-", label="COs", ms=10, color=CO_color)
    ax.plot(bins[:-1], d["co_exp_freqs"], ".--", label="COs (expected)", ms=5, color=CO_color, alpha=0.5)
    ax.plot(bins[:-1], d["nco_freqs"], ".-", label="NCOs", ms=10, color=NCO_color)
    ax.plot(bins[:-1], d["nco_exp_freqs"], ".--", label="NCOs (expected)", ms=5, color=NCO_color, alpha=0.5)
    ax.set_xticks(bins[:-1], labels=[f"{int(x / 1e6)}" for x in bins[:-1]])
    ax.set_xlabel("Distance to telomere (Mb)")
    ax.set_title(f"{label} dataset")
    ax.spines[["right", "top"]].set_visible(False)
axs[0].set_ylabel("Proportion")
axs[0].legend()
fig.tight_layout()
fig.savefig(figure_dir / "subtelomeric_distances_by_dataset.pdf")
subtelomere_summary


## PRDM9 position


In [ ]:
def calculate_motif_distance_to_converted_snps_histogram(reads_df, motif_center_column, motif_strand_column, signal_column=None, max_dist=30000):
    H = np.zeros(max_dist * 2 + 1)
    xs = np.arange(-max_dist, max_dist + 1)

    n_rows = 0
    for row in reads_df.iter_rows(named=True):
        n_rows += 1
        background_allele = row["high_quality_snp_positions_alleles"][0]
        n_converted_alleles = len([x for x in row["high_quality_snp_positions_alleles"] if x != background_allele])
        w = 1.0 if signal_column is None else row[signal_column]

        for snp_pos, snp_allele in zip(row["high_quality_snp_positions"], row["high_quality_snp_positions_alleles"]):
            if snp_allele != background_allele:
                weight = w / n_converted_alleles
                dist_to_motif = snp_pos + row["grch38_reference_start"] - row[motif_center_column]
                if row[motif_strand_column] == 1:
                    H[dist_to_motif - (-max_dist)] += weight
                else:
                    H[-dist_to_motif - (-max_dist)] += weight
    H /= n_rows
    return xs, H


def motif_distance_histogram_symmetry_permutation_testing(
    reads_df,
    motif_center_column,
    motif_strand_column,
    signal_column=None,
    max_dist=30000,
    n_perms=10000,
    hist_func=calculate_motif_distance_to_converted_snps_histogram,
    stat="max_sq_cumsum",
):
    def symm_stat(H):
        H1 = H
        H2 = H[::-1]
        if stat == "max_abs":
            return np.max(np.abs(H1 - H2))
        if stat == "sum_abs":
            return np.sum(np.abs(H1 - H2))
        if stat == "max_abs_cumsum":
            return np.max(np.abs(np.cumsum(H1) - np.cumsum(H2)))
        if stat == "sum_sq_cumsum":
            return np.sum(np.square(np.cumsum(H1) - np.cumsum(H2)))
        if stat == "max_sq_cumsum":
            return np.max(np.square(np.cumsum(H1) - np.cumsum(H2)))

    rng = np.random.default_rng(1)
    S = symm_stat(hist_func(reads_df, motif_center_column, motif_strand_column, signal_column, max_dist)[1])

    permed = []
    for _ in range(n_perms):
        permed.append(symm_stat(hist_func(
            reads_df.with_columns(
                pl.Series(name=motif_strand_column, values=rng.integers(2, size=len(reads_df)))
            ),
            motif_center_column,
            motif_strand_column,
            signal_column,
            max_dist,
        )[1]))
    permed = np.array(permed)
    return np.mean(permed >= S)


def grenander(data, p, k):
    n = len(data)
    data = np.sort(data).astype(float)
    diff = data[k:] - data[:-k]
    tot = data[k:] + data[:-k]
    if np.any(diff == 0):
        return np.nanmean(np.where(diff == 0, tot, np.nan)) / 2
    b = (tot / diff**p).sum() / 2
    a = (1 / diff**p).sum()
    return b / a


def estimate_mode(data, p=10, k=None):
    if k is None:
        k = int(len(data) / 2)
    return grenander(data, p, k)


In [ ]:
NCO_motif_df = NCO_clean_df.filter(pl.col("AA_motif_center_pos").is_not_null())
xs, H = calculate_motif_distance_to_converted_snps_histogram(
    NCO_motif_df,
    "AA_motif_center_pos",
    "AA_motif_strand",
)

n_reads = len(NCO_motif_df)
est_mode = estimate_mode(np.concatenate([[x] * int(cnt) for x, cnt in zip(xs, H * n_reads) if cnt]))
est_median = np.median(np.concatenate([[x] * int(cnt) for x, cnt in zip(xs, H * n_reads) if cnt]))

z = len(xs) // 2
center_xs = xs[z - 2000:z + 2000]
center_H = H[z - 2000:z + 2000]
center_H /= center_H.sum()
center_counts = np.concatenate([[x] * int(cnt) for x, cnt in zip(center_xs, center_H * n_reads) if cnt])
center_mode = estimate_mode(center_counts, p=50, k=100)
center_median = np.median(center_counts)
center_mean_abs = np.sum(np.abs(center_xs) * center_H)
center_upstream = center_H[center_xs < 0].sum()

motif_symmetry_p = motif_distance_histogram_symmetry_permutation_testing(
    NCO_motif_df,
    "AA_motif_center_pos",
    "AA_motif_strand",
    max_dist=30000,
    n_perms=10000,
)

fig, ax = plt.subplots(figsize=(3, 3))
ax.plot(
    xs,
    scipy.ndimage.uniform_filter1d(H, 100),
    color=NCO_color,
    label="NCO",
)
ax.set_xlim(-500, 500)
ax.axvline(0, color="black", ls="--", lw=0.5, alpha=1)
ax.axvline(est_median, color=NCO_color, ls="--")
ax.spines[["right", "top"]].set_visible(False)
ax.set_xlabel("Distance to PRDM9 motif (bp)")
ax.set_ylabel("Proportion")
ax.legend()
fig.tight_layout()
fig.savefig(figure_dir / "nco_converted_snv_prdm9_motif_relative_position.pdf")


## Summary


In [ ]:
near_telomere_fisher_p = scipy.stats.fisher_exact([
    [high_qual_CO_telomere_counts[0], high_qual_CO_telomere_counts.sum() - high_qual_CO_telomere_counts[0]],
    [high_qual_NCO_telomere_counts[0], high_qual_NCO_telomere_counts.sum() - high_qual_NCO_telomere_counts[0]],
]).pvalue

summary = {
    "single-SNP NCOs": int((NCO_tract_df["n_converted"] == 1).sum()),
    "total NCOs": len(NCO_tract_df),
    "single-SNP fraction": float((NCO_tract_df["n_converted"] == 1).mean()),
    "max converted markers": int(NCO_tract_df["n_converted"].max()),
    "single geometric mean bp": 71,
    "single geometric CI low bp": float(single_ci[0, 1]),
    "single geometric CI high bp": float(single_ci[1, 1]),
    "mixture short mean bp": 31,
    "mixture long mean bp": 1220,
    "mixture short fraction": 0.993,
    "mixture long fraction": 0.007,
    "mixture short CI low bp": float(mixture_ci[0, 1]),
    "mixture short CI high bp": float(mixture_ci[1, 1]),
    "mixture long CI low bp": float(mixture_ci[0, 2]),
    "mixture long CI high bp": float(mixture_ci[1, 2]),
    "near-telomere CO count": int(high_qual_CO_telomere_counts[0]),
    "near-telomere NCO count": int(high_qual_NCO_telomere_counts[0]),
    "near-telomere Fisher P": near_telomere_fisher_p,
    "NCO motif reads": len(NCO_motif_df),
    "NCO motif converted SNPs": int(round(H.sum() * len(NCO_motif_df))),
    "mean abs distance to motif bp": float(center_mean_abs),
    "upstream fraction": float(center_upstream),
    "median distance to motif bp": float(center_median),
    "motif symmetry permutation P": float(motif_symmetry_p),
}
summary
